# Fitbit Fitness Tracker Data - Simple EDA

## Step 0: Setup reference

Environment and input checks are recorded in
[Project_Setup.ipynb](Project_Setup.ipynb).

This notebook will include its own minimal imports and path configuration
so it can run independently of the setup notebook's Python session.

## Step 1: Analysis brief

The purpose, questions, metrics, and analysis rules are recorded in
[analysis_brief.md](analysis_brief.md).

## Step 2: Load and inventory the files

Check each file's structure before exploring its values.
Keep the two exports separate and leave raw files unchanged.

In [19]:
import sys
from pathlib import Path

# Find the project root 
cwd = Path.cwd().resolve() 

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / ".git").exists()
        and (folder / "requirements.txt").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

# Stop if the notebook uses the wrong environment.
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project's .venv kernel.")


DATA_ROOT = PROJECT_ROOT / "Dataset"
MANIFEST_PATH = PROJECT_ROOT / "EDA" / "input_manifest.csv"  

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset folder is missing: {DATA_ROOT}")

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Input manifest is missing: {MANIFEST_PATH}")

# Import pandas after checking the environment.
import pandas as pd  

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_ROOT)
print("Manifest    :", MANIFEST_PATH)
print("Project environment: OK")

Project root: D:\Projects\AI and ML\FitBit Fitness Tracker Data
Data folder : D:\Projects\AI and ML\FitBit Fitness Tracker Data\Dataset
Manifest    : D:\Projects\AI and ML\FitBit Fitness Tracker Data\EDA\input_manifest.csv
Project environment: OK


### 2.1 File inventory

Start with the saved input manifest. Row counts, columns, and other
file details will be checked next.

In [5]:
# Read the saved file list, not the raw datasets 
manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig",
)

# Convert file sizes to numbers.
manifest["size_bytes"] = pd.to_numeric(
    manifest["size_bytes"],
    errors="raise",
)

summary = manifest.groupby("source_folder").agg(
    files=("source_file", "size"),
    size_bytes=("size_bytes", "sum"),
)

summary["size_mb"] = (summary["size_bytes"] / 1_000_000).round(1)

print("Files listed:", len(manifest))
display(summary)

Files listed: 29


,files,size_bytes,size_mb
source_folder,,,
P1,11,278798829,278.8
P2,18,337997716,338.0


#### Check the input files

Check each file's existence, size, and SHA-256 before loading its data.
Stop if anything differs from the saved manifest.

In [10]:
import hashlib 

checked_files = 0 

# Check if there are any extra CSVs not included in this analysis 
on_disk = {
    p.relative_to(DATA_ROOT).as_posix()
    for p in DATA_ROOT.rglob("*.csv")
    if p.is_file()
}

extra = sorted(on_disk - set(manifest["source_file"]))

if extra: 
    print("Extra CSVs - not included in this analysis:") 
    for name in extra: 
         print(name) 
else: 
    print("No extra CSV files found.") 

for entry in manifest.itertuples(index=False):
    path = DATA_ROOT / entry.source_file

    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {entry.source_file}")

    if path.stat().st_size != entry.size_bytes:
        raise ValueError(f"File size changed: {entry.source_file}")

    # Hash the file without loading the table into memory.
    with path.open("rb") as file:
        actual_hash = hashlib.file_digest(file, "sha256").hexdigest()

    if actual_hash != entry.sha256:
        raise ValueError(f"Checksum changed: {entry.source_file}")

    checked_files += 1

print(f"Verified {checked_files} files: sizes and checksums match.")

No extra CSV files found.
Verified 29 files: sizes and checksums match.


#### Read the column names

Record each file's columns and check for repeated or empty column names.

In [11]:
import csv 
 
header_records = []

for entry in manifest.itertuples(index=False):
    path = DATA_ROOT / entry.source_file 

    # Read only the header, keeping the original names. 
    with path.open(encoding="utf-8-sig", newline="") as file: 
        reader = csv.reader(file, strict=True)
        columns = next(reader, None)  

    if columns is None: 
        raise ValueError(f"File is empty: {enter.source_file}") 


    # Record the columns and flag possible naming problems.
    header_records.append({
        "source_folder": entry.source_folder,
        "source_file": entry.source_file,
        "file_name": path.name,
        "column_count": len(columns),
        "columns": tuple(columns),
        "repeated_names": len(columns) != len(set(columns)),
        "empty_names": any(not name.strip() for name in columns),
        "padded_names": any(name != name.strip() for name in columns),
    })

header_inventory = pd.DataFrame(header_records) 

# Show the checks without displaying participant data.
display(header_inventory[[
    "source_folder",
    "file_name",
    "column_count",
    "repeated_names",
    "empty_names",
    "padded_names",
]])  

,source_folder,file_name,column_count,repeated_names,empty_names,padded_names
0,P1,dailyActivity_merged.csv,15,False,False,False
1,P1,heartrate_seconds_merged.csv,3,False,False,False
2,P1,hourlyCalories_merged.csv,3,False,False,False
3,P1,hourlyIntensities_merged.csv,4,False,False,False
4,P1,hourlySteps_merged.csv,3,False,False,False
5,P1,minuteCaloriesNarrow_merged.csv,3,False,False,False
6,P1,minuteIntensitiesNarrow_merged.csv,3,False,False,False
7,P1,minuteMETsNarrow_merged.csv,3,False,False,False
8,P1,minuteSleep_merged.csv,4,False,False,False
9,P1,minuteStepsNarrow_merged.csv,3,False,False,False


#### Compare headers across exports

Check whether matching filenames have the same column names and order.
This alone does not decide whether their data can be combined.

In [12]:
comparison_records = []

# Compare files that share the same filename.
for file_name, group in header_inventory.groupby("file_name"):
    columns_by_export = (
        group.set_index("source_folder")["columns"].to_dict()
    )

    if "P1" in columns_by_export and "P2" in columns_by_export:
        p1_columns = columns_by_export["P1"]
        p2_columns = columns_by_export["P2"]

        # Tuples compare names and order; sets compare names only.
        if p1_columns == p2_columns:
            result = "Same names and order"
        elif set(p1_columns) == set(p2_columns):
            result = "Same names, different order"
        else:
            result = "Different column names"
    else:
        result = "One export only"

    comparison_records.append({
        "file_name": file_name,
        "exports": ", ".join(sorted(columns_by_export)),
        "header_check": result,
    })

header_comparison = pd.DataFrame(comparison_records)

display(header_comparison)

,file_name,exports,header_check
0,dailyActivity_merged.csv,"P1, P2",Same names and order
1,dailyCalories_merged.csv,P2,One export only
2,dailyIntensities_merged.csv,P2,One export only
3,dailySteps_merged.csv,P2,One export only
4,heartrate_seconds_merged.csv,"P1, P2",Same names and order
5,hourlyCalories_merged.csv,"P1, P2",Same names and order
6,hourlyIntensities_merged.csv,"P1, P2",Same names and order
7,hourlySteps_merged.csv,"P1, P2",Same names and order
8,minuteCaloriesNarrow_merged.csv,"P1, P2",Same names and order
9,minuteCaloriesWide_merged.csv,P2,One export only


The comparison shows:
- 11 filenames appear in both exports, with identical column names and order.
- 7 appear only in P2.
- None appear only in P1.

#### Count rows and check record length

Read one file at a time. Exclude the header from row counts.
Count and flag empty records or records with the wrong number of columns.

In [13]:

def count_file_rows(path):
    rows = 0
    blank_rows = 0
    wrong_column_count = 0

    # Read records one at a time, without loading the full table.
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file, strict=True)
        header = next(reader)
        expected_columns = len(header)

        for record in reader:
            rows += 1

            # Count empty records without skipping them.
            if not record:
                blank_rows += 1

            # Flag records whose length differs from the header.
            if len(record) != expected_columns:
                wrong_column_count += 1

    return rows, blank_rows, wrong_column_count

row_records = []

for entry in manifest.itertuples(index=False):
    path = DATA_ROOT / entry.source_file

    # Stop and name the file if CSV parsing fails.
    try:
        rows, blank_rows, wrong_columns = count_file_rows(path)
    except csv.Error as error:
        raise ValueError(
            f"CSV parsing failed: {entry.source_file}: {error}"
        ) from error

    row_records.append({
        "source_folder": entry.source_folder,
        "source_file": entry.source_file,
        "file_name": path.name,
        "rows": rows,
        "blank_rows": blank_rows,
        "wrong_column_count": wrong_columns,
    })

row_inventory = pd.DataFrame(row_records)

# Display counts only, without participant data.
display(row_inventory[[
    "source_folder",
    "file_name",
    "rows",
    "blank_rows",
    "wrong_column_count",
]])
 

,source_folder,file_name,rows,blank_rows,wrong_column_count
0,P1,dailyActivity_merged.csv,457,0,0
1,P1,heartrate_seconds_merged.csv,1154681,0,0
2,P1,hourlyCalories_merged.csv,24084,0,0
3,P1,hourlyIntensities_merged.csv,24084,0,0
4,P1,hourlySteps_merged.csv,24084,0,0
5,P1,minuteCaloriesNarrow_merged.csv,1445040,0,0
6,P1,minuteIntensitiesNarrow_merged.csv,1445040,0,0
7,P1,minuteMETsNarrow_merged.csv,1445040,0,0
8,P1,minuteSleep_merged.csv,198559,0,0
9,P1,minuteStepsNarrow_merged.csv,1445040,0,0


### 2.2 Group files by their columns

Files belong to the same header group when they have the same
column names, regardless of order.

In [18]:
# Sort names so column order does not affect grouping.
column_sets = header_inventory["columns"].map(
    lambda names: tuple(sorted(names))
)

# Give matching column sets the same group number.
header_inventory["header_group"] = (
    pd.factorize(column_sets, sort=False)[0] + 1
)

# Summarize which files and exports belong to each group.
header_groups = header_inventory.groupby("header_group").agg(
    files=("source_file", "size"),
    column_count=("column_count", "first"),
    exports=(
        "source_folder",
        lambda values: ", ".join(sorted(set(values))),
    ),
    file_names=(
        "file_name",
        lambda values: ", ".join(sorted(set(values))),
    ),
)

display(header_groups)

,files,column_count,exports,file_names
header_group,,,,
1,2,15,"P1, P2",dailyActivity_merged.csv
2,2,3,"P1, P2",heartrate_seconds_merged.csv
3,2,3,"P1, P2",hourlyCalories_merged.csv
4,2,4,"P1, P2",hourlyIntensities_merged.csv
5,2,3,"P1, P2",hourlySteps_merged.csv
6,2,3,"P1, P2",minuteCaloriesNarrow_merged.csv
7,2,3,"P1, P2",minuteIntensitiesNarrow_merged.csv
8,2,3,"P1, P2",minuteMETsNarrow_merged.csv
9,2,4,"P1, P2",minuteSleep_merged.csv


### 2.3 Small previews

Read three records to understand the structure.
Keep values as text and replace person IDs with preview-only codes.

These records are not a representative sample.
Codes such as P01 are not stable across files.

In [21]:
# Keep original text and avoid automatic conversions.
READ_POLICY = {
    "sep": ",",
    "encoding": "utf-8-sig",
    "encoding_errors": "strict",
    "dtype": "string",
    "na_filter": False,
    "parse_dates": False,
    "skip_blank_lines": False,
    "on_bad_lines": "error",
}

# Start with the first file.
first_file = header_inventory.iloc[0]
path = DATA_ROOT / first_file["source_file"]

# Read only three records.
preview = pd.read_csv(path, nrows=3, **READ_POLICY)

# Change only the display copy.
display_preview = preview.copy()

# Choose the person-ID columns for this preview.
PERSON_ID_COLUMNS = ["Id"]

for column in PERSON_ID_COLUMNS:
    if column in display_preview.columns:
        values = display_preview[column]

        # Give codes only to IDs that are present.
        present_ids = values[values.notna() & values.ne("")]
        codes = {
            value: f"P{i + 1:02d}"
            for i, value in enumerate(pd.unique(present_ids))
        }

        # Keep empty IDs unchanged.
        display_preview[column] = values.replace(codes)

print("Export:", first_file["source_folder"])
print("File  :", first_file["file_name"])

# Show fields vertically for easier reading.
display(display_preview.T)

Export: P1
File  : dailyActivity_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityDate,3/25/2016,3/26/2016,3/27/2016
TotalSteps,11004,17609,12736
TotalDistance,7.1100001335144,11.5500001907349,8.52999973297119
TrackerDistance,7.1100001335144,11.5500001907349,8.52999973297119
LoggedActivitiesDistance,0,0,0
VeryActiveDistance,2.5699999332428,6.92000007629395,4.65999984741211
ModeratelyActiveDistance,0.46000000834465,0.730000019073486,0.159999996423721
LightActiveDistance,4.07000017166138,3.91000008583069,3.71000003814697
SedentaryActiveDistance,0,0,0


The preview worked:
- All three records share the same ID code.
- These dates use month/day/year.
- Long decimals and zero values remain unchanged.
- Units and the meaning of Calories still need checking.

In [32]:
def show_preview(file_info, person_id_columns, hidden_columns=()):
    path = DATA_ROOT / file_info["source_file"]

    # Read three records without changing their text.
    sample = pd.read_csv(path, nrows=3, **READ_POLICY)
    shown = sample.copy()

    # Replace person IDs only in the display copy.
    for column in person_id_columns:
        if column in shown.columns:
            values = shown[column]
            present_ids = values[values.notna() & values.ne("")]

            codes = {
                value: f"P{i + 1:02d}"
                for i, value in enumerate(pd.unique(present_ids))
            }

            shown[column] = values.replace(codes)



    # Log IDs may link records, so hide them in shared previews.
    # Keep empty values unchanged so missing data stays visible.    
    for column in hidden_columns:
        if column in shown.columns:
            values = shown[column]
            shown[column] = values.mask(
                values.notna() & values.ne(""),
                "[hidden]",
            )

    print("Export:", file_info["source_folder"])
    print("File  :", file_info["file_name"])

    # Display fields vertically.
    display(shown.T)

In [33]:
# Choose one file from the second header group.
next_file = header_inventory.loc[
    header_inventory["header_group"].eq(2)
].iloc[0]

show_preview(next_file, person_id_columns=["Id"])

Export: P1
File  : heartrate_seconds_merged.csv


,0,1,2
Id,P01,P01,P01
Time,4/1/2016 7:54:00 AM,4/1/2016 7:54:05 AM,4/1/2016 7:54:10 AM
Value,93,91,96


This preview shows:
- All three records share the same ID code.
- Time includes seconds and AM/PM.
- These three timestamps are 5 seconds apart. That doesn’t establish the spacing throughout the file.
- The meaning and unit of Value still need checking.

In [34]:
# Preview one file from each hourly header group.
for group_number in [3, 4, 5]:
    file_info = header_inventory.loc[
        header_inventory["header_group"].eq(group_number)
    ].iloc[0]

    # Read three rows and replace person IDs with preview codes.
    show_preview(file_info, person_id_columns=["Id"])

Export: P1
File  : hourlyCalories_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,3/12/2016 12:00:00 AM,3/12/2016 1:00:00 AM,3/12/2016 2:00:00 AM
Calories,48,48,48


Export: P1
File  : hourlyIntensities_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,3/12/2016 12:00:00 AM,3/12/2016 1:00:00 AM,3/12/2016 2:00:00 AM
TotalIntensity,0,0,0
AverageIntensity,0,0,0


Export: P1
File  : hourlySteps_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,3/12/2016 12:00:00 AM,3/12/2016 1:00:00 AM,3/12/2016 2:00:00 AM
StepTotal,0,0,0


The hourly previews look consistent with our reading rules:
- ActivityHour includes a date and a 12-hour time.
- The steps and intensity previews contain zeros; we keep them as recorded.
- The calories preview contains 48; its meaning still needs checking.
Remember: P01 is assigned separately in each preview. It doesn’t prove these files show the same person.

In [35]:
# Preview minute calories, intensities, METs, and steps.
for group_number in [6, 7, 8, 10]:
    file_info = header_inventory.loc[
        header_inventory["header_group"].eq(group_number)
    ].iloc[0]

    # Keep original values and use preview-only person codes.
    show_preview(file_info, person_id_columns=["Id"])

Export: P1
File  : minuteCaloriesNarrow_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityMinute,3/12/2016 12:00:00 AM,3/12/2016 12:01:00 AM,3/12/2016 12:02:00 AM
Calories,0.797299981117249,0.797299981117249,0.797299981117249


Export: P1
File  : minuteIntensitiesNarrow_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityMinute,3/12/2016 12:00:00 AM,3/12/2016 12:01:00 AM,3/12/2016 12:02:00 AM
Intensity,0,0,0


Export: P1
File  : minuteMETsNarrow_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityMinute,3/12/2016 12:00:00 AM,3/12/2016 12:01:00 AM,3/12/2016 12:02:00 AM
METs,10,10,10


Export: P1
File  : minuteStepsNarrow_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityMinute,3/12/2016 12:00:00 AM,3/12/2016 12:01:00 AM,3/12/2016 12:02:00 AM
Steps,0,0,0


These samples contain decimal calorie values and METs values of 10.
Meanings, units, and scaling still need checking.

In [36]:
# Check sleep and weight headers before displaying records.
for group_number in [9, 11]:
    file_info = header_inventory.loc[
        header_inventory["header_group"].eq(group_number)
    ].iloc[0]

    print("File:", file_info["file_name"])
    print("Columns:", list(file_info["columns"]))
    print()

File: minuteSleep_merged.csv
Columns: ['Id', 'date', 'value', 'logId']

File: weightLogInfo_merged.csv
Columns: ['Id', 'Date', 'WeightKg', 'WeightPounds', 'Fat', 'BMI', 'IsManualReport', 'LogId']



In [41]:
# Use the exact log-ID spelling found in each file.
for group_number, log_column in [(9, "logId"), (11, "LogId")]:
    file_info = header_inventory.loc[
        header_inventory["header_group"].eq(group_number)
    ].iloc[0]

    show_preview(
        file_info,
        person_id_columns=["Id"],
        hidden_columns=[log_column],
    )

Export: P1
File  : minuteSleep_merged.csv


,0,1,2
Id,P01,P01,P01
date,3/13/2016 2:39:30 AM,3/13/2016 2:40:30 AM,3/13/2016 2:41:30 AM
value,1,1,1
logId,[hidden],[hidden],[hidden]


Export: P1
File  : weightLogInfo_merged.csv


,0,1,2
Id,P01,P02,P03
Date,4/5/2016 11:59:59 PM,4/10/2016 6:33:26 PM,4/3/2016 11:59:59 PM
WeightKg,53.2999992370605,129.600006103516,63.4000015258789
WeightPounds,117.506384062611,285.719105247709,139.773077589277
Fat,22,,10
BMI,22.9699993133545,46.1699981689453,24.7700004577637
IsManualReport,True,False,True
LogId,[hidden],[hidden],[hidden]


These previews show a few useful details:
- Sleep timestamps include seconds ending in :30.
- Sleep value is 1 in these rows; its meaning is still unknown.
- Fat has an empty field. We’ll include "" in its missing-token list.
- IsManualReport shows True and False, currently stored as text.
- The ID masking keeps empty values visible.

In [43]:
# Preview daily calories, intensities, steps, and sleep.
for group_number in [12, 13, 14, 18]:
    file_info = header_inventory.loc[
        header_inventory["header_group"].eq(group_number)
    ].iloc[0]

    # Use preview-only person codes.
    show_preview(file_info, person_id_columns=["Id"])

Export: P2
File  : dailyCalories_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityDay,4/12/2016,4/13/2016,4/14/2016
Calories,1985,1797,1776


Export: P2
File  : dailyIntensities_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityDay,4/12/2016,4/13/2016,4/14/2016
SedentaryMinutes,728,776,1218
LightlyActiveMinutes,328,217,181
FairlyActiveMinutes,13,19,11
VeryActiveMinutes,25,21,30
SedentaryActiveDistance,0,0,0
LightActiveDistance,6.05999994277954,4.71000003814697,3.91000008583069
ModeratelyActiveDistance,0.550000011920929,0.689999997615814,0.400000005960464
VeryActiveDistance,1.87999999523163,1.57000005245209,2.44000005722046


Export: P2
File  : dailySteps_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityDay,4/12/2016,4/13/2016,4/14/2016
StepTotal,13162,10735,10460


Export: P2
File  : sleepDay_merged.csv


,0,1,2
Id,P01,P01,P01
SleepDay,4/12/2016 12:00:00 AM,4/13/2016 12:00:00 AM,4/15/2016 12:00:00 AM
TotalSleepRecords,1,2,1
TotalMinutesAsleep,327,384,412
TotalTimeInBed,346,407,442


These previews show two date formats:
- ActivityDay: date only.
- SleepDay: date plus time, shown at midnight here. That doesn’t establish when sleep started.
- TotalSleepRecords includes 2; we still need its definition.

In [44]:
# Keep the wide previews short: show the first and last fields.
with pd.option_context("display.max_rows", 12, "display.min_rows", 12):
    for group_number in [15, 16, 17]:
        file_info = header_inventory.loc[
            header_inventory["header_group"].eq(group_number)
        ].iloc[0]

        # Read three records and replace person IDs.
        show_preview(file_info, person_id_columns=["Id"])

Export: P2
File  : minuteCaloriesWide_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,4/13/2016 12:00:00 AM,4/13/2016 1:00:00 AM,4/13/2016 2:00:00 AM
Calories00,1.88759994506836,0.786499977111816,0.786499977111816
Calories01,2.20219993591309,0.786499977111816,0.786499977111816
Calories02,0.94379997253418,0.786499977111816,0.786499977111816
Calories03,0.94379997253418,0.786499977111816,0.786499977111816
...,...,...,...
Calories54,2.35949993133545,0.786499977111816,0.786499977111816
Calories55,1.88759994506836,0.786499977111816,0.786499977111816
Calories56,0.94379997253418,0.786499977111816,0.786499977111816


Export: P2
File  : minuteIntensitiesWide_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,4/13/2016 12:00:00 AM,4/13/2016 1:00:00 AM,4/13/2016 2:00:00 AM
Intensity00,1,0,0
Intensity01,1,0,0
Intensity02,0,0,0
Intensity03,0,0,0
...,...,...,...
Intensity54,1,0,0
Intensity55,1,0,0
Intensity56,0,0,0


Export: P2
File  : minuteStepsWide_merged.csv


,0,1,2
Id,P01,P01,P01
ActivityHour,4/13/2016 12:00:00 AM,4/13/2016 1:00:00 AM,4/13/2016 2:00:00 AM
Steps00,4,0,0
Steps01,16,0,0
Steps02,0,0,0
Steps03,0,0,0
...,...,...,...
Steps54,20,0,0
Steps55,1,0,0
Steps56,0,0,0


The wide files have 62 columns. The previews show ActivityHour and
numbered fields from 00 to 59. These suggest minute slots; verify
before reshaping or comparing with narrow files.

### 2.4 Check and parse dates

Map each date column to a format seen in the previews.
Then check that format against the full files, keeping the original text.

In [47]:
# Formats seen in previews; full-file checks come next.
date_only = "%m/%d/%Y"
date_and_time = "%m/%d/%Y %I:%M:%S %p"

DATE_FORMATS = {
    "ActivityDate": date_only,
    "ActivityDay": date_only,
    "ActivityHour": date_and_time,
    "ActivityMinute": date_and_time,
    "Time": date_and_time,
    "date": date_and_time,
    "Date": date_and_time,
    "SleepDay": date_and_time,
}

date_records = []

# Find the date columns in each file.
for file_info in header_inventory.itertuples(index=False):
    matched_columns = [
        column for column in file_info.columns
        if column in DATE_FORMATS
    ]

    if not matched_columns:
        raise ValueError(f"No date column mapped: {file_info.file_name}")

    # Keep the file reference for the full-file checks.
    for column in matched_columns:
        date_records.append({
            "source_folder": file_info.source_folder,
            "source_file": file_info.source_file,
            "file_name": file_info.file_name,
            "date_column": column,
            "format": DATE_FORMATS[column],
        })

date_plan = pd.DataFrame(date_records)

# Show the plan without reading data records.
display(date_plan[[
    "source_folder", "file_name", "date_column", "format"
]])

,source_folder,file_name,date_column,format
0,P1,dailyActivity_merged.csv,ActivityDate,%m/%d/%Y
1,P1,heartrate_seconds_merged.csv,Time,%m/%d/%Y %I:%M:%S %p
2,P1,hourlyCalories_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
3,P1,hourlyIntensities_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
4,P1,hourlySteps_merged.csv,ActivityHour,%m/%d/%Y %I:%M:%S %p
5,P1,minuteCaloriesNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
6,P1,minuteIntensitiesNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
7,P1,minuteMETsNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p
8,P1,minuteSleep_merged.csv,date,%m/%d/%Y %I:%M:%S %p
9,P1,minuteStepsNarrow_merged.csv,ActivityMinute,%m/%d/%Y %I:%M:%S %p


In [49]:
# Start with the first file in the date plan.
rule = date_plan.iloc[0]
path = DATA_ROOT / rule["source_file"]

# Read only the date column, keeping its original text.
data = pd.read_csv(
    path,
    usecols=[rule["date_column"]],
    **READ_POLICY,
)

raw_dates = data[rule["date_column"]]

# Keep empty dates separate from parsing failures.
non_empty = raw_dates.notna() & raw_dates.ne("")

# Parse into a separate Series; leave the original text unchanged.
parsed_dates = pd.to_datetime(
    raw_dates,
    format=rule["format"],
    exact=True,
    errors="coerce",
)

failed = non_empty & parsed_dates.isna()
valid = non_empty & parsed_dates.notna()

eligible_count = int(non_empty.sum())
failed_count = int(failed.sum())

# Report NA when there are no eligible dates.
failure_percent = (
    round(100 * failed_count / eligible_count, 2)
    if eligible_count else pd.NA
)

first_date_report = pd.DataFrame([{
    "source_folder": rule["source_folder"],
    "file_name": rule["file_name"],
    "rows": len(data),
    "empty_dates": int((~non_empty).sum()),
    "eligible_dates": eligible_count,
    "parsed_dates": int(valid.sum()),
    "failed_dates": failed_count,
    "failure_percent": failure_percent,

    # Full date range for this file, using valid dates only.
    "min_date": parsed_dates[valid].min(),
    "max_date": parsed_dates[valid].max(),
}])

display(first_date_report)

# Show up to five different failures to help find the cause.
if failed_count > 0:
    print("Failed date examples — up to 5:")

    for value in raw_dates[failed].drop_duplicates().head(5):
        # Show quotes and spaces clearly.
        print(repr(value))

,source_folder,file_name,rows,empty_dates,eligible_dates,parsed_dates,failed_dates,failure_percent,min_date,max_date
0,P1,dailyActivity_merged.csv,457,0,457,457,0,0.0,2016-03-12,2016-04-12


For P1 daily activity:
- All 457 dates parsed.
- No empty dates or parsing failures.
- Range: March 12–April 12, 2016.

In [50]:
def check_file_dates(rule):
    path = DATA_ROOT / rule.source_file

    counts = {
        "rows": 0,
        "empty_dates": 0,
        "eligible_dates": 0,
        "parsed_dates": 0,
        "failed_dates": 0,
    }

    minimums = []
    maximums = []
    examples = []

    # Read only the date column, in small chunks.
    with pd.read_csv(
        path,
        usecols=[rule.date_column],
        chunksize=100_000,
        **READ_POLICY,
    ) as chunks:
        for chunk in chunks:
            raw = chunk[rule.date_column]
            non_empty = raw.notna() & raw.ne("")

            # Parse separately; keep the original text unchanged.
            parsed = pd.to_datetime(
                raw,
                format=rule.format,
                exact=True,
                errors="coerce",
            )

            failed = non_empty & parsed.isna()
            valid_dates = parsed[non_empty & parsed.notna()]

            # Add this chunk's counts.
            counts["rows"] += len(chunk)
            counts["empty_dates"] += int((~non_empty).sum())
            counts["eligible_dates"] += int(non_empty.sum())
            counts["parsed_dates"] += len(valid_dates)
            counts["failed_dates"] += int(failed.sum())

            # Keep only the date bounds from each chunk.
            if not valid_dates.empty:
                minimums.append(valid_dates.min())
                maximums.append(valid_dates.max())

            # Keep at most five different failing values per file.
            if len(examples) < 5:
                for value in raw[failed].drop_duplicates():
                    if value not in examples:
                        examples.append(value)
                    if len(examples) == 5:
                        break

    eligible = counts["eligible_dates"]

    return {
        "source_folder": rule.source_folder,
        "source_file": rule.source_file,
        "file_name": rule.file_name,
        "date_column": rule.date_column,
        **counts,
        "failure_percent": (
            round(100 * counts["failed_dates"] / eligible, 2)
            if eligible else pd.NA
        ),
        "min_date": min(minimums, default=pd.NaT),
        "max_date": max(maximums, default=pd.NaT),
        "failed_examples": examples,
    }


# Check that this pass reads the previously counted number of rows.
expected_rows = row_inventory.set_index("source_file")["rows"].to_dict()
date_results = []

for rule in date_plan.itertuples(index=False):
    result = check_file_dates(rule)

    if result["rows"] != expected_rows[rule.source_file]:
        raise ValueError(f"Row count differs: {rule.file_name}")

    date_results.append(result)

    # Print examples only when parsing fails.
    if result["failed_examples"]:
        print(rule.source_folder, rule.file_name)
        for value in result["failed_examples"]:
            print(repr(value))

date_report = pd.DataFrame(date_results)

# Show counts and ranges without participant records.
display(date_report.drop(columns=["source_file", "failed_examples"]))

,source_folder,file_name,date_column,rows,empty_dates,eligible_dates,parsed_dates,failed_dates,failure_percent,min_date,max_date
0,P1,dailyActivity_merged.csv,ActivityDate,457,0,457,457,0,0.0,2016-03-12 00:00:00,2016-04-12 00:00:00
1,P1,heartrate_seconds_merged.csv,Time,1154681,0,1154681,1154681,0,0.0,2016-03-29 00:00:05,2016-04-12 11:03:15
2,P1,hourlyCalories_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
3,P1,hourlyIntensities_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
4,P1,hourlySteps_merged.csv,ActivityHour,24084,0,24084,24084,0,0.0,2016-03-12 00:00:00,2016-04-12 10:00:00
5,P1,minuteCaloriesNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
6,P1,minuteIntensitiesNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
7,P1,minuteMETsNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00
8,P1,minuteSleep_merged.csv,date,198559,0,198559,198559,0,0.0,2016-03-11 21:19:30,2016-04-12 08:35:00
9,P1,minuteStepsNarrow_merged.csv,ActivityMinute,1445040,0,1445040,1445040,0,0.0,2016-03-12 00:00:00,2016-04-12 10:59:00


### 2.5 Observed date ranges

The ranges come from full-file checks, not samples.

Important findings:
- Daily activity includes April 12 in both exports.
  This does not yet prove duplicate participant records.
- P1 minute sleep starts on March 11; P2 minute sleep starts on April 11,
  before their folder start labels.
- P2 wide files span April 13-May 13.
  P2 narrow minute files span April 12-May 12.
  The reason for this difference is not yet known.
- Coverage differs by file. For example, P1 heart-rate data starts
  on March 29, while daily activity starts on March 12.

Keep the observed timestamps unchanged.
Do not trim or shift dates to match folder labels.

Successful parsing does not prove that timestamps are correctly aligned.
A minimum and maximum do not prove continuous coverage.
The source timezone remains unknown.

**2.6: File-set differences are covered by the header comparison: 7 files exist only in P2 (structural absence, not missing values).**

In [51]:
file_info = header_inventory.iloc[0] 
path = DATA_ROOT / file_info["source_file"]  
sample = pd.read_csv(path, nrows=3, **READ_POLICY) 

dictionary_records = [] 

for column in sample.columns: 
    values = sample[column]

    present = values[vlaues.notna() & values.ne("")]
    example = (
          present.ilco[0]
    ) 

           Id ActivityDate TotalSteps     TotalDistance   TrackerDistance  \
0  1503960366    3/25/2016      11004   7.1100001335144   7.1100001335144   
1  1503960366    3/26/2016      17609  11.5500001907349  11.5500001907349   
2  1503960366    3/27/2016      12736  8.52999973297119  8.52999973297119   

  LoggedActivitiesDistance VeryActiveDistance ModeratelyActiveDistance  \
0                        0    2.5699999332428         0.46000000834465   
1                        0   6.92000007629395        0.730000019073486   
2                        0   4.65999984741211        0.159999996423721   

  LightActiveDistance SedentaryActiveDistance VeryActiveMinutes  \
0    4.07000017166138                       0                33   
1    3.91000008583069                       0                89   
2    3.71000003814697                       0                56   

  FairlyActiveMinutes LightlyActiveMinutes SedentaryMinutes Calories  
0                  12                  205            